# Aktivitas Naive Bayes — Apakah Cocok Bermain Tenis Hari Ini?

**Mata Kuliah:** DSB07 — Machine Learning  
**Topik:** Naive Bayes (Pertemuan 10)  
**Durasi:** 45 menit  

## Anggota Kelompok 1

| No | Nama                         | NIM      |
|---:|------------------------------|----------|
| 1  | Hendrawan Wijaya             | 32230055 |
| 2  | Khetta Ajnatavindriya Likito | 32230059 |
| 3  | Calvin Ang                   | 32230067 |
| 4  | Ferdinand Arya Wijaya        | 32230071 |
| 5  | Yosua Imanuel Widjaja        | 32230073 |
| 6  | Jonathan Kelvin Haslim       | 32230077 |
| 7  | Yehezkiel Petra Kairupan     | 32230079 |


## Petunjuk
Lengkapi setiap sel berlabel `# TODO`. Jangan ubah sel yang sudah berisi kode lengkap kecuali diminta.
Jalankan sel secara berurutan dari atas ke bawah.

## Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

pd.set_option("display.precision", 4)
np.set_printoptions(precision=4, suppress=True)

## Tahap 1 — Memahami Data (5 menit)

Dataset `play_tennis` berisi 14 hari pengamatan apakah cocok untuk bermain tenis berdasarkan kondisi cuaca.

| Fitur | Nilai mungkin |
|-------|---------------|
| Outlook | Sunny, Overcast, Rain |
| Temperature | Hot, Mild, Cool |
| Humidity | High, Normal |
| Wind | Weak, Strong |
| **Play** (label) | **Yes, No** |

In [2]:
# Ganti USERNAME dengan akun GitHub dosen Anda setelah repo di-push.
url = "https://raw.githubusercontent.com/UBM-ML/naive-bayes/main/data/play_tennis.csv"
df = pd.read_csv(url)
df

,Outlook,Temperature,Humidity,Wind,Play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [3]:
# TODO 1: Tampilkan distribusi label 'Play' (berapa Yes, berapa No)
# Hint: gunakan .value_counts() pada kolom 'Play'
print(df["Play"].value_counts())

Play
Yes    9
No     5
Name: count, dtype: int64


## Tahap 2 — Hitung Manual (15 menit)

**Kasus uji:** Hari ini cuaca **Sunny**, suhu **Cool**, kelembapan **High**, angin **Strong**.  
**Pertanyaan:** Apakah cocok bermain tenis (`Play = Yes` atau `No`)?

Rumus (slide 2.2):
$$P(K \mid F_1, F_2, \dots, F_n) \propto P(K) \times \prod_{i=1}^{n} P(F_i \mid K)$$

Kita akan menghitung *tanpa* Laplace smoothing terlebih dahulu (sesuai rumus di slide).

In [4]:
test = {"Outlook": "Sunny", "Temperature": "Cool", "Humidity": "High", "Wind": "Strong"}
test

{'Outlook': 'Sunny',
 'Temperature': 'Cool',
 'Humidity': 'High',
 'Wind': 'Strong'}

### 2.1 Probabilitas Prior P(K)

In [6]:
# TODO 2: Hitung P(Yes) dan P(No)
n_total = len(df)                         # total baris
n_yes   = (df["Play"] == "Yes").sum()     # jumlah Play == 'Yes'
n_no    = (df["Play"] == "No").sum()      # jumlah Play == 'No'

p_yes = n_yes / n_total
p_no  = n_no / n_total

print(f"P(Yes) = {p_yes:.4f}")
print(f"P(No)  = {p_no:.4f}")

P(Yes) = 0.6429
P(No)  = 0.3571


### 2.2 Probabilitas Kondisional P(F_i | K)

Untuk setiap fitur dalam kasus uji, hitung berapa kali nilainya muncul dalam tiap kelas, dibagi total baris kelas tersebut.

In [7]:
df_yes = df[df["Play"] == "Yes"]
df_no  = df[df["Play"] == "No"]

# TODO 3: Hitung 4 probabilitas kondisional untuk kelas Yes
# Contoh: p_sunny_yes = (df_yes['Outlook'] == 'Sunny').sum() / len(df_yes)
p_sunny_yes  = (df_yes["Outlook"] == "Sunny").sum() / len(df_yes)
p_cool_yes   = (df_yes["Temperature"] == "Cool").sum() / len(df_yes)
p_high_yes   = (df_yes["Humidity"] == "High").sum() / len(df_yes)
p_strong_yes = (df_yes["Wind"] == "Strong").sum() / len(df_yes)

# TODO 4: Hitung 4 probabilitas kondisional untuk kelas No
p_sunny_no  = (df_no["Outlook"] == "Sunny").sum() / len(df_no)
p_cool_no   = (df_no["Temperature"] == "Cool").sum() / len(df_no)
p_high_no   = (df_no["Humidity"] == "High").sum() / len(df_no)
p_strong_no = (df_no["Wind"] == "Strong").sum() / len(df_no)

print("Kelas Yes:", p_sunny_yes, p_cool_yes, p_high_yes, p_strong_yes)
print("Kelas No :", p_sunny_no,  p_cool_no,  p_high_no,  p_strong_no)

Kelas Yes: 0.2222222222222222 0.3333333333333333 0.3333333333333333 0.3333333333333333
Kelas No : 0.6 0.2 0.8 0.6


### 2.3 Posterior & Keputusan

In [8]:
# TODO 5: Hitung skor posterior (proporsional, belum dinormalisasi)
score_yes = p_yes * p_sunny_yes * p_cool_yes * p_high_yes * p_strong_yes
score_no  = p_no  * p_sunny_no  * p_cool_no  * p_high_no  * p_strong_no

# TODO 6: Normalisasi sehingga total = 1
total    = score_yes + score_no
post_yes = score_yes / total
post_no  = score_no / total

prediksi_manual = "Yes" if post_yes > post_no else "No"

print(f"P(Yes | x) = {post_yes:.4f}")
print(f"P(No  | x) = {post_no:.4f}")
print(f"Prediksi manual: {prediksi_manual}")

P(Yes | x) = 0.2046
P(No  | x) = 0.7954
Prediksi manual: No


## Tahap 3 — Implementasi dengan scikit-learn (15 menit)

`CategoricalNB` di sklearn membutuhkan input numerik, jadi fitur kategorikal kita ubah dulu dengan `OrdinalEncoder`.

In [9]:
features = ["Outlook", "Temperature", "Humidity", "Wind"]

encoder = OrdinalEncoder()
X = encoder.fit_transform(df[features])
y = df["Play"].values

print("Pemetaan kategori -> angka:")
for col, cats in zip(features, encoder.categories_):
  print(f"  {col}: {dict(enumerate(cats))}")

Pemetaan kategori -> angka:
  Outlook: {0: 'Overcast', 1: 'Rain', 2: 'Sunny'}
  Temperature: {0: 'Cool', 1: 'Hot', 2: 'Mild'}
  Humidity: {0: 'High', 1: 'Normal'}
  Wind: {0: 'Strong', 1: 'Weak'}


In [10]:
# TODO 7: Latih CategoricalNB (alpha default = 1, pakai Laplace smoothing)
model = CategoricalNB()
model.fit(X, y)

# TODO 8: Hitung akurasi pada data latih (perlu .predict(X) dulu)
y_pred  = model.predict(X)
akurasi = accuracy_score(y, y_pred)
print(f"Akurasi pada data latih: {akurasi:.4f}")

Akurasi pada data latih: 0.9286


In [11]:
# TODO 9: Prediksi kasus uji yang SAMA dengan Tahap 2
x_test_df = pd.DataFrame([test])
x_test = encoder.transform(x_test_df[features])

prediksi_sklearn = model.predict(x_test)
proba_sklearn    = model.predict_proba(x_test)

print(f"Prediksi sklearn : {prediksi_sklearn[0]}")
print(f"Kelas-kelas      : {model.classes_}")
print(f"Probabilitas     : {proba_sklearn[0]}")

Prediksi sklearn : No
Kelas-kelas      : ['No' 'Yes']
Probabilitas     : [0.7201 0.2799]


## Tahap 4 — Diskusi Kelompok (7 menit)

_Tulis jawaban kelompok dalam sel di bawah._

1. Apakah **prediksi kelas** manual sama dengan sklearn? Apakah **nilai probabilitas**-nya juga sama? Jika berbeda, kira-kira kenapa?  
   _Hint: cari arti parameter `alpha` di `CategoricalNB` (Laplace / additive smoothing)._
2. Coba ubah `CategoricalNB(alpha=1e-10)` dan jalankan ulang Tahap 3. Apakah probabilitas sklearn jadi lebih dekat ke hasil manual?
3. Apa yang akan terjadi jika kasus uji mengandung kombinasi (kategori, kelas) yang **tidak pernah muncul** di data latih (misalnya `Outlook=Overcast` pada kelas `No`)? Bagaimana Laplace smoothing menyelamatkan situasi ini?  
   _Hint: lihat slide 1.3 poin #3 (Sensitif terhadap Atribut yang Hilang)._
4. Naive Bayes mengasumsikan **semua fitur saling independen**. Menurut kelompok Anda, apakah `Humidity` dan `Outlook` benar-benar independen di dunia nyata? Apa konsekuensinya untuk akurasi model?

**Jawaban Kelompok:**

1. Hasil prediksi kelas sama (No), tetapi nilai probabilitasnya berbeda (manual = 0.7954; sklearn = 0.7201). Hal ini dikarenakan sklearn menggunakan *Laplace smoothing* (`alpha=1`) yang "menghaluskan" nilai probabilitas.
2. Ya, dengan menggunakan `alpha=1e-10`, hasil sklearn identik dengan perhitungan manual (manual = 0.7954; sklearn = 0.7954). ***(lihat potongan kode di bawah)***
3. Tanpa smoothing, kombinasi yang tidak pernah muncul di data latih akan menghasilkan probabilitas kondisional = 0. Karena Naive Bayes mengalikan semua probabilitas kondisional, satu nilai nol saja akan membuat seluruh skor posterior menjadi 0. *Laplace smoothing* menyelamatkan situasi ini dengan menambahkan nilai `alpha` (biasanya 1) ke hitungan setiap kombinasi, termasuk yang tidak pernah muncul. Akibatnya, tidak ada probabilitas yang bernilai nol, dan model tetap bisa berfungsi untuk kasus-kasus yang belum pernah dilihat sebelumnya.
4. `Humidity` dan `Outlook` tidak independen di dunia nyata. Secara logis, cuaca `Sunny` cenderung berkorelasi dengan `Humidity = Normal` (udara kering), sedangkan `Rain` cenderung berkorelasi dengan `Humidity = High`. Ada hubungan kausal yang nyata antara kedua fitur ini. Konsekuensinya, asumsi independen Naive Bayes dilanggar, sehingga model menghitung probabilitas gabungan seolah-olah fitur-fitur tersebut tidak saling mempengaruhi, padahal sebenarnya ada. Hal ini dapat menyebabkan *over-confident predictions* (probabilitas posterior terlalu ekstrem) dan potensial menurunkan akurasi, terutama pada kasus-kasus di mana korelasi antarfitur sangat kuat. Meski demikian, Naive Bayes sering tetap memberikan hasil yang cukup baik bahkan ketika asumsi ini tidak sepenuhnya terpenuhi.

In [12]:
# Coba CategoricalNB dengan alpha=1e-10
model = CategoricalNB(alpha=1e-10)
model.fit(X, y)

y_pred  = model.predict(X)
akurasi = accuracy_score(y, y_pred)
print(f"Akurasi pada data latih: {akurasi:.4f}")

Akurasi pada data latih: 0.9286


In [13]:
# Lakukan prediksi kasus uji yang SAMA
x_test_df = pd.DataFrame([test])
x_test = encoder.transform(x_test_df[features])

prediksi_sklearn = model.predict(x_test)
proba_sklearn    = model.predict_proba(x_test)

print(f"Prediksi sklearn : {prediksi_sklearn[0]}")
print(f"Kelas-kelas      : {model.classes_}")
print(f"Probabilitas     : {proba_sklearn[0]}")

Prediksi sklearn : No
Kelas-kelas      : ['No' 'Yes']
Probabilitas     : [0.7954 0.2046]


## Tahap 5 — Refleksi (3 menit)

Tulis 3 kalimat singkat:
- Hal terpenting yang saya pelajari hari ini adalah cara mengembangkan model klasifikasi Naive Bayes.
- Bagian paling sulit adalah memahami cara kerja rumus/perhitungan manual dari Naive Bayes.
- Naive Bayes cocok dipakai ketika jumlah data latih yang digunakan terbatas. Selain itu, model ini juga cocok untuk kebutuhan klasifikasi yang cepat dan ringan.

## Submission
Simpan notebook ini ke Google Drive masing-masing, lalu kumpulkan link **share** (akses *Anyone with the link → Viewer*) ke kanal kelas yang ditentukan dosen.